In [1]:
#!pip install torch torchvision torchaudio
from torchvision import datasets, transforms
from torch.utils.data import Subset, DataLoader
import numpy as np
from pathlib import Path
import torch.nn.functional as F

home = Path('/data')
train_dir = home / 'datasets/imagenet_full_size/061417/train'
test_dir = home / 'datasets/imagenet_full_size/061417/val'


normalize = transforms.Normalize(mean=[0.485, 0.456, 0.406],
                                 std=[0.229, 0.224, 0.225])

transform_train = transforms.Compose([
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.AutoAugment(policy=transforms.AutoAugmentPolicy.IMAGENET),
    transforms.ToTensor(),
    normalize,
])

transform_test = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    normalize,
])


trainset = datasets.ImageFolder(train_dir, transform=transform_train)
testset = datasets.ImageFolder(test_dir, transform=transform_test)



# Sample 10% of trainset
# subset_size = int(0.1 * len(trainset))
# indices = np.random.choice(len(trainset), subset_size, replace=False)
# train_subset = Subset(trainset, indices)

train_loader = DataLoader(trainset, batch_size=64, shuffle=False, num_workers=4)
test_loader = DataLoader(testset, batch_size=64, shuffle=False, num_workers=4)


In [2]:

import sklearn
import torch
from train import ResNet18

# Set up device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Recreate model architecture
n_classes = 1000  # or whatever your original number was
lda_args = None   # or use the same lda_args you trained with

checkpoint = torch.load("models/deeplda_best-3.pth", map_location=device)

# Rebuild model
embedding_model = ResNet18(num_classes=n_classes, lda_args=lda_args)
embedding_model.load_state_dict(checkpoint['state_dict'])
embedding_model.to(device)
embedding_model.eval()




ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (shortcut): Sequential()
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (conv2): Conv2d(64, 64, kernel_

In [3]:
from tqdm import tqdm

def extract_embeddings(dataloader, model, device, desc="Extracting"):
    model.eval()
    embeddings, labels = [], []
    with torch.no_grad():
        for x, y in tqdm(dataloader, desc=desc):
            x = x.to(device)
            y = y.to(device)
            feats = model._forward_impl(x)
            feats = F.normalize(feats, p=2, dim=1)  # L2 normalization
            embeddings.append(feats)
            labels.append(y)
    return torch.cat(embeddings), torch.cat(labels)

X_train, y_train = extract_embeddings(train_loader, embedding_model, device, desc="Train Embeddings")
X_test, y_test = extract_embeddings(test_loader, embedding_model, device, desc="Test Embeddings")


Test Embeddings: 100%|██████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████████| 782/782 [00:59<00:00, 13.15it/s]


In [4]:


from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import accuracy_score


# Initialize the LDA model
lda = LinearDiscriminantAnalysis(solver='lsqr', shrinkage='auto')  # Optional: you can tweak parameters

# Fit the LDA model
lda.fit(X_train.cpu().numpy(), y_train.cpu().numpy())

# Make predictions on the test set
y_pred = lda.predict(X_test.cpu().numpy())

# Calculate accuracy
acc = accuracy_score(y_test.cpu().numpy(), y_pred)
print(f"Test Accuracy: {acc * 100:.2f}%")


Test Accuracy: 24.52%


In [12]:
# import torch
# import torch.nn.functional as F
# from sklearn.metrics import accuracy_score

# # --- Compute class centroids (normalized) ---
# def compute_normalized_centroids_torch(X, y):
#     classes = torch.unique(y)
#     centroids = []
#     for cls in classes:
#         class_feats = X[y == cls]
#         mean = class_feats.mean(dim=0, keepdim=True)
#         mean = F.normalize(mean, p=2, dim=1)
#         centroids.append(mean)
#     return torch.cat(centroids, dim=0), classes

# # --- Predict based on cosine similarity ---
# def predict_cosine_torch(X, centroids, classes):
#     X = F.normalize(X, p=2, dim=1)
# #     sims = torch.matmul(X, centroids.T)
# #     pred_indices = sims.argmax(dim=1)
# #     return classes[pred_indices]

# # # --- Main ---
# # # Make sure these are torch tensors on the right device
# # X_train_torch = X_train.to(device)
# # y_train_torch = y_train.to(device)
# # X_test_torch = X_test.to(device)
# # y_test_torch = y_test.to(device)

# # centroids, class_labels = compute_normalized_centroids_torch(X_train_torch, y_train_torch)
# # y_pred_torch = predict_cosine_torch(X_test_torch, centroids, class_labels)

# # # Convert to CPU and NumPy for scoring
# # acc = accuracy_score(y_test.cpu().numpy(), y_pred_torch.cpu().numpy())
# # print(f"Centroid Cosine Classifier Accuracy: {acc * 100:.2f}%")

# model_state = embedding_model.state_dict()
# checkpoint_state = checkpoint['state_dict']

# model_keys = set(model_state.keys())
# checkpoint_keys = set(checkpoint_state.keys())




# for name in model_state:
#     if name in checkpoint_state:
#         if not torch.allclose(model_state[name], checkpoint_state[name], atol=1e-6):
#             print(f"Value mismatch in {name}")


In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader

class LinearClassifier(nn.Module):
    def __init__(self, input_dim, num_classes):
        super().__init__()
        self.linear = nn.Linear(input_dim, num_classes)

    def forward(self, x):
        return self.linear(x)

# --- Config ---
batch_size = 1024
lr = 1e-3
epochs = 200
device = 'cuda' if torch.cuda.is_available() else 'cpu'

# --- Prepare data ---
train_ds = TensorDataset(X_train, y_train)
train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)

test_ds = TensorDataset(X_test, y_test)
test_loader = DataLoader(test_ds, batch_size=batch_size)

# --- Model, Loss, Optimizer ---
model = LinearClassifier(X_train.shape[1], int(y_train.max().item()) + 1).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=lr)
criterion = nn.CrossEntropyLoss()

# --- Training ---
for epoch in range(epochs):
    model.train()
    correct, total = 0, 0
    for xb, yb in train_loader:
        xb, yb = xb.to(device), yb.to(device)
        out = model(xb)
        loss = criterion(out, yb)

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        preds = out.argmax(dim=1)
        correct += (preds == yb).sum().item()
        total += yb.size(0)

    acc = correct / total * 100
    print(f"Epoch {epoch+1}: Train Accuracy = {acc:.2f}%")

    # --- Evaluation ---
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for xb, yb in test_loader:
            xb, yb = xb.to(device), yb.to(device)
            preds = model(xb).argmax(dim=1)
            correct += (preds == yb).sum().item()
            total += yb.size(0)

    test_acc = correct / total * 100
    print(f"Test Accuracy = {test_acc:.2f}%")


Epoch 1: Train Accuracy = 15.83%
Test Accuracy = 25.48%
Epoch 2: Train Accuracy = 21.27%
Test Accuracy = 26.99%
Epoch 3: Train Accuracy = 22.44%
Test Accuracy = 28.23%
Epoch 4: Train Accuracy = 23.34%
Test Accuracy = 29.30%
Epoch 5: Train Accuracy = 23.96%
Test Accuracy = 30.11%
Epoch 6: Train Accuracy = 24.42%
Test Accuracy = 30.68%
Epoch 7: Train Accuracy = 24.79%
Test Accuracy = 31.04%
Epoch 8: Train Accuracy = 25.03%
Test Accuracy = 31.47%
Epoch 9: Train Accuracy = 25.24%
Test Accuracy = 31.76%
Epoch 10: Train Accuracy = 25.43%
Test Accuracy = 31.91%
Epoch 11: Train Accuracy = 25.58%
Test Accuracy = 32.11%
Epoch 12: Train Accuracy = 25.72%
Test Accuracy = 32.31%
Epoch 13: Train Accuracy = 25.84%
Test Accuracy = 32.41%
Epoch 14: Train Accuracy = 25.95%
Test Accuracy = 32.63%
Epoch 15: Train Accuracy = 26.05%
Test Accuracy = 32.71%
Epoch 16: Train Accuracy = 26.13%
Test Accuracy = 32.83%
Epoch 17: Train Accuracy = 26.22%
Test Accuracy = 32.86%
Epoch 18: Train Accuracy = 26.29%
Test A

In [5]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
import torch.nn.functional as F

# ------------------------
# LDA Module
# ------------------------
class LDA(nn.Module):
    def __init__(self, n_classes, lamb):
        super().__init__()
        self.n_classes = n_classes
        self.n_components = n_classes - 1
        self.lamb = lamb

    def _lda(self, X, y):
        X = X.view(X.shape[0], -1)
        N, D = X.shape
        labels, counts = torch.unique(y, return_counts=True)
        assert len(labels) == self.n_classes

        X_bar = X - X.mean(0)
        Xc_mean = torch.zeros((self.n_classes, D), device=X.device)
        St = X_bar.T @ X_bar / (N - 1)
        Sw = torch.zeros((D, D), device=X.device)

        for c, Nc in zip(labels, counts):
            Xc = X[y == c]
            mean_c = Xc.mean(0)
            Xc_mean[c] = mean_c
            Xc_bar = Xc - mean_c
            Sw += (Xc_bar.T @ Xc_bar) / (Nc - 1)
        Sw /= self.n_classes
        Sb = St - Sw
        Sw += torch.eye(D, device=X.device) * self.lamb

        eigvals, eigvecs = torch.linalg.eig(Sw.pinverse() @ Sb)
        eigvals = eigvals.real
        eigvecs = eigvecs.real

        eigvals, idx = eigvals.sort()
        eigvecs = eigvecs[:, idx]

        hasComplex = eigvecs.shape[1] < eigvecs.shape[0]
        return hasComplex, Xc_mean, eigvals, eigvecs

    def forward(self, X, y):
        hasComplex, Xc_mean, evals, evecs = self._lda(X, y)
        self.scalings_ = evecs
        self.coef_ = Xc_mean @ evecs @ evecs.T
        self.intercept_ = -0.5 * torch.diagonal(Xc_mean @ self.coef_.T)
        return hasComplex, evals

    def transform(self, X):
        return X @ self.scalings_#[:, -self.n_components:]

    def predict(self, X):
        logits = X @ self.coef_.T + self.intercept_
        return logits.argmax(dim=1)

# ------------------------
# Training Function
# ------------------------
def train_lda_classifier_with_dataloader(X_train, y_train, X_test, y_test, n_classes, lamb=1e-4, epochs=50, lr=0.1, batch_size=128):
    device = X_train.device
    lda_model = LDA(n_classes=n_classes, lamb=lamb).to(device)

    hasComplex, _ = lda_model(X_train, y_train)
    if hasComplex:
        raise ValueError("Complex eigenvalues encountered in LDA")

    # Project embeddings with LDA
    X_train_proj = lda_model.transform(X_train).detach()
    X_test_proj = lda_model.transform(X_test).detach()

    # Dataloaders for mini-batch training
    train_ds = TensorDataset(X_train_proj, y_train)
    test_ds = TensorDataset(X_test_proj, y_test)
    train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
    test_loader = DataLoader(test_ds, batch_size=batch_size)

    # Linear classifier
    clf = nn.Linear(X_train_proj.shape[1], n_classes).to(device)
    optimizer = torch.optim.SGD(clf.parameters(), lr=lr)
    criterion = nn.CrossEntropyLoss()

    # Training loop
    for epoch in range(1, epochs + 1):
        clf.train()
        total_loss = 0.0
        for xb, yb in train_loader:
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            logits = clf(xb)
            loss = criterion(logits, yb)
            loss.backward()
            optimizer.step()
            total_loss += loss.item() * xb.size(0)

        # Accuracy evaluation
        clf.eval()
        def accuracy(loader):
            correct, total = 0, 0
            with torch.no_grad():
                for xb, yb in loader:
                    preds = clf(xb.to(device)).argmax(dim=1)
                    correct += (preds == yb.to(device)).sum().item()
                    total += xb.size(0)
            return correct / total

        train_acc = accuracy(train_loader)
        test_acc = accuracy(test_loader)

        print(f"Epoch {epoch:02d} | Loss: {total_loss/len(train_ds):.4f} | Train Acc: {train_acc*100:.2f}% | Test Acc: {test_acc*100:.2f}%")

    return lda_model, clf

# ------------------------
# Run it
# ------------------------
n_classes = len(torch.unique(y_train))
lda_model, clf = train_lda_classifier_with_dataloader(
    X_train, y_train, X_test, y_test,
    n_classes=n_classes,
    lamb=1e-4,
    epochs=3,
    lr=1e-1,
    batch_size=4096
)






Epoch 01 | Loss: 6.9065 | Train Acc: 0.15% | Test Acc: 0.14%
Epoch 02 | Loss: 6.9031 | Train Acc: 0.39% | Test Acc: 0.46%
Epoch 03 | Loss: 6.8998 | Train Acc: 1.05% | Test Acc: 1.24%


In [6]:
lda_model = LDA(n_classes=n_classes, lamb=1e-4).to(device)

hasComplex, _ = lda_model(X_train, y_train)
if hasComplex:
    raise ValueError("Complex eigenvalues encountered in LDA")

# Project embeddings with LDA
pred = lda_model.predict(X_test)
100 * (y_test == pred).sum() / len(pred)


tensor(27.1840, device='cuda:0')